# Árboles de Decisión para Regresión _(versión detallada)_

_Intuición, partición recursiva y comparación con la regresión lineal_

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

---
> 📖 **Nota:** Esta es la versión detallada del notebook `03_arboles_decision_regresion.ipynb`.  
> Incluye teoría ampliada y comentarios línea a línea en cada bloque de código.

![Aprendizaje Supervisado](assets/header.png)

## 1. Intuición — preguntas tipo "sí/no"

Un árbol de decisión es básicamente una secuencia de **preguntas binarias** sobre los predictores. A diferencia de la regresión lineal (que asume una relación lineal global), el árbol **divide el espacio de los datos en regiones** y hace predicciones locales.

### Estructura del árbol

```
                    [Nodo raíz]
                   ¿GrLivArea ≤ 1500?
                  /                  \
               SÍ                     NO
              /                        \
    [Nodo interno]              [Nodo interno]
   ¿OverallQual ≤ 6?          ¿YearBuilt ≤ 1980?
      /        \                  /          \
    SÍ         NO               SÍ           NO
    /           \               /             \
[Hoja]       [Hoja]         [Hoja]        [Hoja]
pred=$120k   pred=$180k     pred=$200k    pred=$280k
```

### Componentes clave

| Componente | Descripción |
|------------|-------------|
| **Nodo raíz** | Primera pregunta que se hace a todos los datos |
| **Nodo interno** | Pregunta intermedia que divide los datos en dos grupos |
| **Hoja (terminal)** | Nodo final que contiene la predicción (media de $y$ de las observaciones que llegaron ahí) |
| **Split** | Punto de corte que divide un nodo en dos hijos |
| **Profundidad** | Número de niveles desde la raíz hasta la hoja más lejana |

### ¿Cómo predice?

Para una nueva observación:
1. Empieza en la raíz
2. Evalúa la condición (e.g., ¿`GrLivArea` ≤ 1500?)
3. Baja por la rama izquierda (SÍ) o derecha (NO)
4. Repite hasta llegar a una hoja
5. Devuelve el valor de esa hoja (la media de $y$ de las observaciones de entrenamiento que cayeron ahí)

> 💡 **Ventaja clave:** Las reglas son **interpretables** — puedes explicar exactamente por qué el modelo predijo cierto valor.

## 2. ¿Qué hace el árbol con el espacio de los datos?

Cada split parte el espacio de los predictores en **regiones rectangulares** (hiper-rectángulos en alta dimensión). En cada región el árbol predice un valor constante: la **media de $y$** de las observaciones de entrenamiento que cayeron en esa región.

### Diferencia con regresión lineal

| Modelo | Forma de la predicción | Flexibilidad |
|--------|----------------------|--------------|
| **Regresión lineal** | Hiperplano suave | Baja — asume relación lineal global |
| **Árbol de decisión** | Regiones rectangulares con valores constantes | Alta — captura no linealidades y interacciones |

### Visualización en 2D

La gráfica siguiente muestra cómo un árbol de profundidad 3 divide el espacio $(x_1, x_2)$ en regiones. Cada región tiene un color que representa la predicción (valor constante). Los bordes son **siempre paralelos a los ejes** (splits ortogonales).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

# Generamos datos sintéticos en 2D con estructura de "escalones"
# para que sea fácil ver cómo el árbol captura patrones no lineales
rng = np.random.default_rng(0)
n = 400

# X tiene 2 features (x1, x2) uniformemente distribuidas en [0, 10]
X = rng.uniform(0, 10, size=(n, 2))

# y tiene una estructura de "escalones" basada en reglas simples:
# - Si x1 < 5 → base = 10, si x1 >= 5 → base = 30
# - Si x2 < 4 → +0,     si x2 >= 4 → +15
# + ruido gaussiano para hacerlo realista
y = (
    np.where(X[:, 0] < 5, 10, 30)      # escalón en x1
    + np.where(X[:, 1] < 4, 0, 15)     # escalón en x2
    + rng.normal(0, 2, n)              # ruido
)

# Entrenamos un árbol de profundidad 3
# max_depth=3 → máximo 2^3 = 8 hojas (regiones)
tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y)

# Creamos una malla (grid) de puntos para visualizar la superficie de predicción
# linspace genera 200 puntos equidistantes entre 0 y 10
# meshgrid crea matrices 2D con todas las combinaciones (x1, x2)
xx, yy = np.meshgrid(np.linspace(0, 10, 200), np.linspace(0, 10, 200))

# Aplanamos la malla en un array 2D de shape (40000, 2) para predecir
grid = np.c_[xx.ravel(), yy.ravel()]

# Predecimos para todos los puntos de la malla
# reshape devuelve la forma original de la malla para graficar
zz = tree.predict(grid).reshape(xx.shape)

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(7, 5))

# contourf dibuja las regiones con colores según el valor predicho
# levels=20 → 20 niveles de color (más suave visualmente)
im = ax.contourf(xx, yy, zz, levels=20, cmap='viridis', alpha=0.85)

# Scatter de los datos reales (color según y verdadero)
ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='white', s=25)

ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Cómo un árbol parte el espacio en regiones rectangulares')
plt.colorbar(im, ax=ax, label='predicción')
plt.show()

# OBSERVACIÓN:
# Las regiones son RECTANGULARES (bordes paralelos a los ejes)
# Cada región tiene un color uniforme (predicción constante)
# El árbol captura la estructura de "escalones" de los datos sintéticos


## 3. ¿Cómo elige cada corte? — Algoritmo CART

El algoritmo **CART** (Classification And Regression Trees) construye el árbol de forma **greedy** (voraz): en cada nodo elige el mejor split posible **en ese momento**, sin mirar hacia adelante.

### Criterio de split para regresión

En cada nodo, el árbol prueba:
- **Todas las variables** $x_j$
- **Todos los puntos de corte** posibles $t$ (valores únicos de $x_j$ en ese nodo)

Y se queda con el split $(j, t)$ que **minimiza la suma de cuadrados de los residuos** en los dos hijos:

$$
\min_{j, t} \left[ \sum_{i \in \text{izq}} (y_i - \bar{y}_{\text{izq}})^2 + \sum_{i \in \text{der}} (y_i - \bar{y}_{\text{der}})^2 \right]
$$

donde $\bar{y}_{\text{izq}}$ y $\bar{y}_{\text{der}}$ son las medias de $y$ en cada hijo.

### Intuición

> _"Quiero que las observaciones dentro de cada hoja se parezcan en $y$ (baja varianza interna)."_

Si todas las casas en una hoja tienen precios similares, la predicción (media de esa hoja) será buena. Si hay mucha dispersión, el error será alto.

### Proceso recursivo

1. **Nodo raíz:** Prueba todos los splits posibles, elige el mejor.
2. **Recursión:** Para cada hijo, repite el proceso (si no se cumple un criterio de parada).
3. **Criterios de parada:**
   - Profundidad máxima alcanzada (`max_depth`)
   - Muy pocas muestras en el nodo (`min_samples_split`)
   - Muy pocas muestras por hoja (`min_samples_leaf`)
   - No hay ganancia suficiente al dividir

### Poda (pruning)

Después de crecer el árbol completo, se puede **podar** (eliminar ramas) para reducir overfitting:
- **Pre-poda:** Parar antes de crecer completamente (hiperparámetros como `max_depth`).
- **Post-poda:** Crecer el árbol completo y luego eliminar ramas que no aportan (parámetro `ccp_alpha` en sklearn).

> ⚠️ **Problema del algoritmo greedy:** No garantiza encontrar el árbol óptimo global, solo el mejor split local en cada paso. Pero es computacionalmente eficiente.

## 4. Árbol vs regresión lineal — comparación detallada

| Aspecto | Regresión Lineal | Árbol de Decisión |
|---------|-----------------|-------------------|
| **Relaciones no lineales** | Requieren transformaciones manuales (e.g., $x^2$, $\log(x)$) | Las captura naturalmente con splits |
| **Interacciones entre variables** | Hay que codificarlas explícitamente (e.g., $x_1 \cdot x_2$) | Implícitas en los splits (un split en $x_1$ puede depender de $x_2$) |
| **Escala de los features** | Sensible — necesita estandarización | Indiferente — solo importa el orden relativo |
| **Interpretabilidad** | Coeficientes con significado estadístico (p-valores) | Reglas if/else legibles, fácil de explicar a no técnicos |
| **Forma de la predicción** | Suave (continua) | Escalonada (constante por regiones) |
| **Riesgo de overfitting** | Bajo–medio (especialmente con regularización) | **Alto sin poda** — puede memorizar el ruido |
| **Extrapolación** | Sí — puede predecir fuera del rango de entrenamiento | **No** — predice constantes (la media de la hoja más cercana) |
| **Estabilidad** | Alta — pequeños cambios en los datos → pequeños cambios en coeficientes | **Baja** — pequeños cambios en los datos pueden cambiar todo el árbol |
| **Número de parámetros** | $p + 1$ (coeficientes) | Depende de la profundidad (puede ser muy grande) |

### Cuándo usar cada uno

**Usa regresión lineal si:**
- La relación es aproximadamente lineal
- Necesitas inferencia estadística (p-valores, intervalos de confianza)
- Quieres extrapolación confiable
- Tienes pocas observaciones

**Usa árbol de decisión si:**
- La relación es claramente no lineal
- Hay interacciones complejas entre variables
- Priorizas interpretabilidad sobre precisión estadística
- Tienes suficientes datos para evitar overfitting
- No necesitas extrapolación

> 💡 **En la práctica:** Los árboles individuales rara vez se usan en producción — se prefieren **ensembles** (Random Forest, Gradient Boosting) que promedian muchos árboles para reducir la varianza.

## 5. Hiperparámetros clave (`DecisionTreeRegressor`)

Los hiperparámetros controlan el **trade-off entre sesgo y varianza**. Un árbol muy profundo (sin restricciones) tiene bajo sesgo pero alta varianza (overfitting). Un árbol muy poco profundo tiene alto sesgo pero baja varianza (underfitting).

### Hiperparámetros principales

| Hiperparámetro | Descripción | Efecto de aumentarlo | Valor típico |
|----------------|-------------|---------------------|--------------|
| `max_depth` | Profundidad máxima del árbol | ↑ Complejidad, ↑ Overfitting | 3-10 |
| `min_samples_split` | Mínimo de muestras para considerar un split | ↓ Complejidad, ↓ Overfitting | 2-20 |
| `min_samples_leaf` | Mínimo de muestras por hoja | ↓ Complejidad, ↓ Overfitting | 1-10 |
| `max_features` | Número de features a considerar en cada split | ↓ Varianza (útil en Random Forest) | `None` (todas) |
| `ccp_alpha` | Parámetro de poda (cost-complexity pruning) | ↑ Poda, ↓ Overfitting | 0.0 (sin poda) |

### Estrategia de ajuste

1. **Empieza simple:** `max_depth=3` o `max_depth=5`.
2. **Evalúa en validación:** Si hay underfitting (R² bajo en train y test) → aumenta `max_depth`.
3. **Si hay overfitting** (R² alto en train, bajo en test):
   - Reduce `max_depth`
   - Aumenta `min_samples_split` o `min_samples_leaf`
   - Usa `ccp_alpha` para poda
4. **Grid search / Random search** para encontrar la mejor combinación.

### Ejemplo de configuraciones

```python
# Árbol simple (bajo overfitting)
DecisionTreeRegressor(max_depth=3, min_samples_leaf=10)

# Árbol complejo (riesgo de overfitting)
DecisionTreeRegressor(max_depth=15, min_samples_split=2)

# Árbol con poda
DecisionTreeRegressor(max_depth=10, ccp_alpha=0.01)
```

> ⚠️ **Importante:** En producción, **siempre valida con datos no vistos** (test set o cross-validation). Un árbol que memoriza el train no sirve.

## 6. Caso práctico — House Prices

Mismo dataset que el notebook 02: cada fila es una vivienda en Ames, Iowa, y queremos predecir `SalePrice`. Usamos solo `housing_train.csv` (el `housing_test.csv` de la competencia no trae la etiqueta) y lo partimos con `train_test_split`.

In [ ]:
from pathlib import Path
import pandas as pd

# Definimos la ruta al CSV usando pathlib (más robusto que strings de ruta)
DATA = Path('../data/housing_train.csv')

# Verificamos que el archivo existe antes de intentar cargarlo
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga housing_train.csv desde '
        'https://www.kaggle.com/c/house-prices-advanced-regression-techniques '
        'y colócalo en data/ (ver README.md).'
    )

# Cargamos el CSV completo en un DataFrame de pandas
# Mismo dataset que en los notebooks 02 (regresión lineal)
# 1460 casas × 81 columnas (79 features + Id + SalePrice)
df = pd.read_csv(DATA)

print('Shape:', df.shape)

# Mostramos las primeras 5 filas para inspeccionar la estructura
df.head()


In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuramos el estilo visual para todos los gráficos
sns.set_theme(style='whitegrid')

# Seleccionamos las mismas 6 features que en el notebook de regresión lineal
# para poder comparar directamente los resultados
features = ['GrLivArea',    # área habitable sobre el suelo (ft²)
            'OverallQual',  # calidad general de la casa (1-10)
            'YearBuilt',    # año de construcción
            'GarageCars',   # capacidad del garage (número de autos)
            'TotalBsmtSF',  # área total del sótano (ft²)
            'FullBath']     # número de baños completos

# Creamos un DataFrame solo con las features seleccionadas + la variable objetivo
# dropna() elimina filas con valores faltantes (NaN)
data = df[features + ['SalePrice']].dropna()

# Separamos features (X) y variable objetivo (y)
X = data[features]
y = data['SalePrice']

# Dividimos en train (80%) y test (20%)
# random_state=42 para reproducibilidad (misma partición que en notebook 02)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# ─────────────────────────────────────────────────────────────────
# Entrenamos ÁRBOL DE DECISIÓN
# ─────────────────────────────────────────────────────────────────
# max_depth=3 → árbol poco profundo (máximo 2^3 = 8 hojas)
# Esto evita overfitting y hace el árbol interpretable (se puede visualizar)
tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_tr, y_tr)

# ─────────────────────────────────────────────────────────────────
# Entrenamos REGRESIÓN LINEAL (para comparar)
# ─────────────────────────────────────────────────────────────────
lr = LinearRegression().fit(X_tr, y_tr)

# ─────────────────────────────────────────────────────────────────
# Función auxiliar para reportar métricas
# ─────────────────────────────────────────────────────────────────
def reporte(nombre, y_true, y_pred):
    """Imprime MAE, RMSE y R² de forma legible."""
    print(f'--- {nombre} ---')
    print(f'  MAE  = {mean_absolute_error(y_true, y_pred):,.0f}')   # error absoluto medio
    print(f'  RMSE = {np.sqrt(mean_squared_error(y_true, y_pred)):,.0f}')  # raíz del error cuadrático medio
    print(f'  R²   = {r2_score(y_true, y_pred):.3f}')  # proporción de varianza explicada

# Evaluamos ambos modelos en el conjunto de test
reporte('Decision Tree (depth=3)', y_te, tree.predict(X_te))
reporte('Regresión lineal',         y_te, lr.predict(X_te))

# INTERPRETACIÓN:
# - Si el árbol tiene R² similar a la regresión lineal → la relación es aproximadamente lineal
# - Si el árbol tiene R² mucho mayor → hay no linealidades que el árbol captura mejor
# - Si el árbol tiene R² menor → puede estar underfitting (aumentar max_depth)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

# plot_tree dibuja el árbol completo con todas sus reglas
# feature_names: nombres de las variables (en lugar de X[0], X[1], etc.)
# filled=True: colorea los nodos según el valor predicho (más oscuro = mayor valor)
# rounded=True: bordes redondeados (más estético)
# precision=0: no muestra decimales en los valores (más legible)
plot_tree(
    tree,
    feature_names=features,
    filled=True,
    rounded=True,
    precision=0,
    ax=ax,
)
ax.set_title('Árbol de regresión (max_depth=3) sobre House Prices')
plt.show()

# CÓMO LEER EL ÁRBOL:
# Cada nodo muestra:
# - Condición del split (e.g., "GrLivArea <= 1792")
# - squared_error: varianza de y en ese nodo (qué tan dispersos están los precios)
# - samples: número de observaciones que llegaron a ese nodo
# - value: predicción (media de SalePrice de las observaciones en ese nodo)
#
# Ejemplo de lectura:
# Si una casa tiene GrLivArea <= 1792 → va a la izquierda
#   Si además OverallQual <= 6 → va a la izquierda de nuevo
#     Predicción final: ~$130k (nodo hoja)
#
# El color indica el valor predicho:
# - Más claro → precio bajo
# - Más oscuro → precio alto


In [ ]:
# --- Importancia de variables ---
# El árbol calcula automáticamente qué tan importante es cada feature
# basándose en cuánto reduce la varianza (squared_error) en los splits

# feature_importances_ es un array con la importancia de cada feature
# Suma 1.0 (100%) — es una proporción relativa
imp = pd.Series(tree.feature_importances_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))

# Gráfico de barras horizontales (más legible para nombres largos)
imp.plot.barh(ax=ax, color='steelblue')
ax.set_title('Importancia de variables — DecisionTreeRegressor')
ax.set_xlabel('Importancia relativa')
plt.show()

# INTERPRETACIÓN:
# - Importancia = 0 → la variable NO se usó en ningún split del árbol
# - Importancia alta → la variable se usó en splits que redujeron mucho la varianza
#
# IMPORTANTE: La importancia es relativa a ESTE árbol específico
# Si entrenas con otra semilla o max_depth diferente, puede cambiar
# Para importancias más robustas, usa Random Forest (promedia muchos árboles)


In [ ]:
# --- ¿Cuánto debería medir el árbol? Curva de complejidad ---
# Entrenamos árboles con diferentes profundidades para ver el trade-off sesgo-varianza

# Probamos profundidades de 1 a 15
depths = range(1, 16)

# Listas para guardar el R² en train y test para cada profundidad
tr_scores, te_scores = [], []

# Para cada profundidad, entrenamos un árbol y evaluamos
for d in depths:
    # Entrenamos un árbol con profundidad d
    m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_tr, y_tr)
    
    # Calculamos R² en train (qué tan bien ajusta los datos de entrenamiento)
    tr_scores.append(r2_score(y_tr, m.predict(X_tr)))
    
    # Calculamos R² en test (qué tan bien generaliza a datos nuevos)
    te_scores.append(r2_score(y_te, m.predict(X_te)))

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(7, 4))

# Línea azul: R² en train (siempre aumenta con la profundidad)
ax.plot(depths, tr_scores, marker='o', label='Train R²', color='blue')

# Línea naranja: R² en test (aumenta hasta cierto punto, luego baja por overfitting)
ax.plot(depths, te_scores, marker='o', label='Test  R²', color='orange')

ax.set_xlabel('max_depth')
ax.set_ylabel('R²')
ax.set_title('Curva de complejidad — overfitting al aumentar la profundidad')
ax.legend()
ax.grid(True)
plt.show()

# INTERPRETACIÓN:
# - Train R² siempre aumenta → el árbol se ajusta mejor a los datos de entrenamiento
# - Test R² aumenta al principio (el modelo aprende patrones reales)
# - Test R² baja después (overfitting — el modelo memoriza ruido del train)
#
# PUNTO ÓPTIMO: donde Test R² es máximo (balance entre sesgo y varianza)
# En este caso, parece estar alrededor de max_depth = 5-7
#
# Si Train R² >> Test R² → overfitting (reducir max_depth o usar poda)
# Si Train R² ≈ Test R² pero ambos bajos → underfitting (aumentar max_depth)


## 7. Referencias

- Breiman, Friedman, Olshen & Stone (1984). *Classification and Regression Trees*.
- ISLR cap. 8: *Tree-Based Methods*.
- scikit-learn user guide: https://scikit-learn.org/stable/modules/tree.html
- Dataset: https://www.kaggle.com/c/house-prices-advanced-regression-techniques

## Predicción sobre datos nuevos — uso del modelo en producción

Una vez que el modelo está validado en el conjunto de test, queremos usarlo para predecir sobre datos que **no hemos visto**. En la práctica seguimos tres pasos:

1. **Reentrenar con todos los datos disponibles.** Ya hicimos la validación con la partición train/test; ahora aprovechamos el 100% de la información para que el modelo final tenga la mejor estimación posible de los parámetros.
2. **Aplicar exactamente las mismas transformaciones** que durante el entrenamiento (mismas columnas, mismo encoding, misma escala) — un error muy común en producción es desalinear el preprocesamiento.
3. **Persistir el modelo** con `joblib` (o `pickle`) para reutilizarlo sin reentrenar.

In [ ]:
import joblib

# ─────────────────────────────────────────────────────────────────
# PASO 1: Reentrenamos el modelo final con TODOS los datos disponibles
# ─────────────────────────────────────────────────────────────────
# Ya validamos el rendimiento con train/test. Ahora usamos el 100% de los datos
# para que el modelo final tenga la mejor estimación posible.
# Esto es estándar en producción: validar con split, entrenar final con todo.

modelo_final = DecisionTreeRegressor(max_depth=3, random_state=42).fit(
    data[features], data['SalePrice']
)

# ─────────────────────────────────────────────────────────────────
# PASO 2: Persistimos el modelo a disco
# ─────────────────────────────────────────────────────────────────
# joblib.dump serializa el objeto Python a un archivo binario
# Es más eficiente que pickle para objetos con arrays grandes (numpy)

joblib.dump(modelo_final, 'modelo_house_tree.pkl')
print('Modelo guardado:', 'modelo_house_tree.pkl')

# Para recuperarlo en otro proceso (API, script batch, notebook diferente):
# modelo_final = joblib.load('modelo_house_tree.pkl')

# IMPORTANTE: En producción real también deberías guardar:
# - La lista de features (orden exacto)
# - Los hiperparámetros usados (max_depth, min_samples_leaf, etc.)
# - Estadísticas de los datos de entrenamiento (para detectar data drift)
# - Versión del modelo y fecha de entrenamiento
# - Métricas de validación (R², RMSE, etc.)


### Inferencia individual — una vivienda hipotética

In [ ]:
# Definimos una vivienda hipotética con todas sus características
# En producción esto vendría de una API (JSON), base de datos, formulario web, etc.
nueva_casa = pd.DataFrame([{
    'GrLivArea':    1700,    # ft² de área habitable sobre el suelo
    'OverallQual':  7,       # calidad general 1-10
    'YearBuilt':    2005,    # año de construcción
    'GarageCars':   2,       # capacidad del garage (número de autos)
    'TotalBsmtSF':  900,     # ft² de sótano
    'FullBath':     2,       # número de baños completos
}])

# CRÍTICO: Las columnas deben estar en el MISMO ORDEN que en el entrenamiento
# Si el orden cambia, el árbol evaluará las condiciones incorrectas
# En producción real, guardarías la lista de features junto al modelo

# predict() devuelve un array con las predicciones
# [0] extrae el primer (y único) elemento
precio = modelo_final.predict(nueva_casa)[0]

print(f'Precio estimado: ${precio:,.0f} USD')

# INTERPRETACIÓN:
# El árbol predice el precio basándose en la HOJA donde cayó esta observación
# La predicción es la MEDIA de los precios de las casas de entrenamiento
# que cayeron en esa misma hoja (tienen características similares)


### Ventaja del árbol: ¿por qué predijo eso?

A diferencia de un modelo lineal, podemos preguntarle al árbol **en qué hoja cayó** la observación y qué reglas se aplicaron. Esto es muy útil para explicar una decisión a un usuario de negocio o a un regulador.

In [ ]:
from sklearn.tree import export_text

# ─────────────────────────────────────────────────────────────────
# VENTAJA CLAVE DE LOS ÁRBOLES: EXPLICABILIDAD
# ─────────────────────────────────────────────────────────────────
# Podemos rastrear exactamente qué reglas se aplicaron para llegar a la predicción

# apply() devuelve el ID de la hoja donde cayó cada observación
# Cada hoja tiene un ID único (número entero)
hoja = modelo_final.apply(nueva_casa)[0]

print(f'La vivienda cayó en la hoja #{hoja}')
print(f'Predicción de esa hoja: ${precio:,.0f}\n')

# export_text() convierte el árbol a reglas legibles en formato texto
# Esto es MUY útil para explicar decisiones a usuarios de negocio o reguladores
print('Reglas del árbol (texto):')
print(export_text(modelo_final, feature_names=list(features)))

# EJEMPLO DE LECTURA:
# Si la salida dice:
# |--- GrLivArea <= 1792.50
# |   |--- OverallQual <= 6.50
# |   |   |--- value: [130000.00]
#
# Significa: "Si el área habitable es ≤ 1792 ft² Y la calidad es ≤ 6.5,
#            entonces el precio estimado es $130,000"
#
# VENTAJA vs regresión lineal:
# - Regresión lineal: "El precio aumenta $54 por cada ft² adicional" (global)
# - Árbol: "Si el área es < 1792 ft², el precio es ~$130k; si es > 1792 ft²
#           y la calidad es alta, el precio es ~$280k" (reglas locales específicas)
#
# Esto es más fácil de explicar a stakeholders no técnicos


### Inferencia en lote sobre el `housing_test.csv` de Kaggle

In [ ]:
test_path = Path('../data/housing_test.csv')

if test_path.exists():
    # Cargamos el CSV de test (1459 casas sin la columna SalePrice)
    test_df = pd.read_csv(test_path)
    
    # Extraemos solo las features que el modelo necesita
    # fillna() rellena valores faltantes con la mediana de cada columna
    # (en producción real, usarías la mediana del TRAIN, no del test)
    X_new = test_df[features].fillna(test_df[features].median(numeric_only=True))
    
    # Predecimos para todas las casas del test de una sola vez (inferencia en lote)
    # Esto es mucho más eficiente que predecir una por una en un loop
    test_df['SalePrice_pred'] = modelo_final.predict(X_new)
    
    # Mostramos las primeras 5 predicciones
    display(test_df[['Id', 'SalePrice_pred']].head())
    
    # Para enviar a Kaggle, bastaría con:
    # submission = test_df[['Id', 'SalePrice_pred']].rename(columns={'SalePrice_pred': 'SalePrice'})
    # submission.to_csv('submission.csv', index=False)
    
else:
    print(f'(Sin {test_path}: salta esta celda. Ver README.md.)')
    
# NOTA: En un sistema de producción real (no competencia Kaggle):
# - Las predicciones se guardarían en una base de datos con timestamp
# - Se loggearían los inputs para auditoría y detección de drift
# - Se monitorizaría la distribución de las predicciones (alertas si cambia mucho)
# - Se rastrearían las hojas donde caen las predicciones (para detectar regiones problemáticas)


## 8. Carga del modelo desde disco — flujo real de producción

En los ejemplos anteriores todo estaba **en memoria** (mismo notebook). En producción el escenario es diferente:

- El modelo se entrenó en un job separado (o en otro servidor) y se guardó como `.pkl`.
- El servicio de inferencia (una API, un script batch, un container) **solo tiene el `.pkl`** — no tiene acceso a los datos de entrenamiento ni a las variables del notebook.

### Flujo real en producción

```
[Entrenamiento]          [Producción / inferencia]
  train.py  ──pkl──►  api.py / batch_job.py / container
                           │
                           1. joblib.load('modelo.pkl')
                           2. recibir datos crudos (JSON, CSV, DB)
                           3. validar que tengan las features correctas
                           4. modelo.predict(X_nuevo)
                           5. (opcional) modelo.apply(X_nuevo) para explicabilidad
                           6. retornar predicción + explicación
```

> 🔑 **Ventaja de los árboles:** Además de la predicción, puedes devolver las **reglas que se aplicaron** (explicabilidad), lo cual es muy valioso en dominios regulados (finanzas, salud, etc.).

In [ ]:
import joblib
import pandas as pd
from sklearn.tree import export_text

# ─────────────────────────────────────────────────────────────────
# PASO 1: Cargar el modelo desde disco
# Esto es lo ÚNICO que necesitas en producción — sin datos de train,
# sin variables previas del notebook.
# ─────────────────────────────────────────────────────────────────
modelo = joblib.load('modelo_house_tree.pkl')

print('Modelo cargado:', type(modelo).__name__)
print('Profundidad del árbol:', modelo.get_depth())
print('Número de hojas:', modelo.get_n_leaves())

# ─────────────────────────────────────────────────────────────────
# PASO 2: Recibir datos crudos (simulamos datos entrantes)
# En producción esto vendría de una API (JSON), base de datos, Kafka, etc.
# ─────────────────────────────────────────────────────────────────
datos_crudos = pd.DataFrame([{
    'GrLivArea':    2500,
    'OverallQual':  9,
    'YearBuilt':    2015,
    'GarageCars':   3,
    'TotalBsmtSF':  1500,
    'FullBath':     3,
}])

# ─────────────────────────────────────────────────────────────────
# PASO 3: Validación de features
# CRÍTICO: Las columnas deben estar en el MISMO ORDEN que en el entrenamiento
# ─────────────────────────────────────────────────────────────────
features_esperadas = ['GrLivArea', 'OverallQual', 'YearBuilt', 
                      'GarageCars', 'TotalBsmtSF', 'FullBath']

# Verificamos que todas las features necesarias estén presentes
if not all(f in datos_crudos.columns for f in features_esperadas):
    raise ValueError(f'Faltan features. Se esperan: {features_esperadas}')

# Seleccionamos solo las features en el orden correcto
X_nuevo = datos_crudos[features_esperadas]

# ─────────────────────────────────────────────────────────────────
# PASO 4: Inferencia + Explicabilidad
# ─────────────────────────────────────────────────────────────────
# Predicción
prediccion = modelo.predict(X_nuevo)[0]

# ID de la hoja donde cayó (útil para debugging y monitoreo)
hoja_id = modelo.apply(X_nuevo)[0]

# Reglas del árbol en formato texto (para explicar la decisión)
reglas = export_text(modelo, feature_names=features_esperadas)

print(f'\n─── RESULTADO ───')
print(f'Precio estimado: ${prediccion:,.0f} USD')
print(f'Hoja del árbol: #{hoja_id}')
print(f'\nReglas aplicadas:')
print(reglas)

# ─────────────────────────────────────────────────────────────────
# BONUS: cómo se vería esto en producción (pseudo-código de una API FastAPI)
# ─────────────────────────────────────────────────────────────────
print("""
─── Ejemplo en una API real (FastAPI) ───────────────────────────

from fastapi import FastAPI
import joblib, pandas as pd
from sklearn.tree import export_text

app = FastAPI()
modelo = joblib.load("modelo_house_tree.pkl")  # carga UNA vez al iniciar
features = ['GrLivArea', 'OverallQual', 'YearBuilt', 
            'GarageCars', 'TotalBsmtSF', 'FullBath']

@app.post("/predict")
def predict(casa: dict):
    df = pd.DataFrame([casa])
    
    # Validar que tenga todas las features
    if not all(f in df.columns for f in features):
        return {"error": "Faltan features"}
    
    X = df[features]  # orden correcto
    precio = modelo.predict(X)[0]
    hoja = modelo.apply(X)[0]
    
    # Explicabilidad: reglas del árbol
    reglas = export_text(modelo, feature_names=features)
    
    return {
        "precio_estimado": round(float(precio), 2),
        "hoja_id": int(hoja),
        "explicacion": reglas  # reglas legibles para el usuario
    }

──────────────────────────────────────────────────────────────────
""")
